# Phase 12a — chunk granularity: does clause-splitting still earn its keep?

The Phase 8 failure analysis found that a single article can crowd every other article out of the context window. Traced concretely on **q22** ("will my pension go up over the years?", ground truth SSL Article 90): six of the ten slots were separate clauses of SSL Article 71, a seventh was Article 71 in Arabic, and Article 90 — sitting at semantic rank 11, one place outside the cutoff — never made it in. Across the eval set, 5 of 40 questions lose ≥3 slots to same-article duplicates, and two of those are outright retrieval failures.

Only 21 of 531 (law, article, language) units are split at all, but they are the ones that matter: SSL Article 71 is 29 pieces in English, Article 64 is 14, Article 85 is 15.

## Three versions compared

| | search index | what fills a slot |
|---|---|---|
| **A — current** | one vector per clause | one clause |
| **B — no splitting** | one vector per whole article | the whole article, always |
| **C — split to search, group to deliver** | one vector per clause | the retrieved clauses of one article, merged |

B and C both hand the generator complete-ish articles, so the only substantive difference between them is **how the index is built**. That isolates the open question: does a single embedding per whole article blur its meaning enough to hurt matching? An embedding must compress its whole text into one fixed vector, so an article covering disability rates, survivor rules and increase percentages at once should in principle match everything vaguely and nothing sharply — but that is an argument, not a measurement, and this project has overturned arguments with data before (see the reranker in Phase 8c, and the weighted-sum-vs-RRF call in Phase 6).

C never alters the stored data. Splitting stays exactly as it is; C simply stops treating each clause as its own slot at the final step, and keeps **only the clauses that actually matched** rather than all 29.

## Scope

**Same-language questions only (32 of 40).** Cross-lingual questions are excluded deliberately: the corpus is parallel, so an Arabic question answered from the Arabic article is a correct answer for a real user — the eval only marked those wrong to stress-test cross-language reach. Numbers here are therefore *not* comparable to the 40-question figures in `ROADMAP.md`. The baseline to beat, scored over these 32: **recall@1 0.844, recall@5 0.938, recall@10 0.969, MRR 0.874**.

Retrieval accuracy alone can't settle this — it only asks whether the right *article* appeared, not whether the generator got the clause containing the answer, nor whether extra bundled text dilutes the response. So this notebook also runs the full pipeline and scores answers with the same independent LLM judge used in Phase 7/8 (Claude Sonnet 5 via OpenRouter).

Nothing here modifies `src/retrieval.py`, the stored chunks, or the Pinecone index. B's article-level vectors are built in memory only. Kernel: `rag311`.

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path(r"c:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from rank_bm25 import BM25Okapi

from src.retrieval import Retriever, RRF_K, RRF_W, BM25_K1, BM25_B, CANDIDATE_TOP_K, COHERE_MODEL, tokenize

retriever = Retriever()
chunks = retriever.chunks

eval_all = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_draft.jsonl", encoding="utf-8")])
LAW_MAP = {"Work_Law_1996": "labor96", "Social_Security_Law": "ssl", "Professional_Work_Law_2019": "prof19"}
eval_all["answer_law_code"] = eval_all["answer_law"].map(LAW_MAP)

# same-language only -- see scope note above
evalq = eval_all[eval_all.question_language == eval_all.answer_language].reset_index(drop=True)
print(f"{len(evalq)} same-language questions (of {len(eval_all)})")

32 same-language questions (of 40)


## Build B's corpus: one unit per whole article

Clauses are concatenated in document order (their original row order), which is the order they appear in the statute.

In [2]:
work = chunks.reset_index().rename(columns={"index": "row_order"})
grouped = (
    work.sort_values("row_order")
        .groupby(["law", "article_number", "language"], as_index=False)
        .agg(text=("text", lambda s: "\n".join(s)),
             n_clauses=("chunk_id", "size"),
             row_order=("row_order", "min"))
        .sort_values("row_order")
        .reset_index(drop=True)
)
grouped["unit_id"] = grouped.apply(lambda r: f"{r['law']}_{r['language']}_{r['article_number']}", axis=1)

print(f"{len(chunks)} clause chunks -> {len(grouped)} whole-article units")
print(f"articles actually split: {(grouped.n_clauses > 1).sum()}")
print(grouped.text.str.len().describe(percentiles=[.5, .9, .99]).round(0).to_string())

739 clause chunks -> 531 whole-article units
articles actually split: 21
count      531.0
mean       899.0
std       1237.0
min         67.0
50%        534.0
90%       1876.0
99%       6135.0
max      13641.0


## Vectors

A and C reuse the clause vectors already in Pinecone (fetched by id — same model, no re-embedding, no cost). B's article vectors are newly embedded with the same Cohere model and held in memory only, so the live index is untouched. All three then score with identical local cosine similarity, so no difference can be attributed to the search backend.

In [3]:
def l2norm(m):
    return m / np.linalg.norm(m, axis=1, keepdims=True)

# --- A/C: pull existing clause vectors out of Pinecone ---
clause_ids = retriever.chunk_ids
vec_by_id = {}
for i in range(0, len(clause_ids), 100):
    res = retriever.index.fetch(ids=clause_ids[i:i + 100])
    vectors = getattr(res, "vectors", None) or res["vectors"]
    for cid, v in vectors.items():
        vec_by_id[cid] = getattr(v, "values", None) or v["values"]

missing = [c for c in clause_ids if c not in vec_by_id]
assert not missing, f"{len(missing)} clause vectors missing from Pinecone"
CLAUSE_M = l2norm(np.array([vec_by_id[c] for c in clause_ids], dtype=np.float32))
print("clause vectors:", CLAUSE_M.shape)

# --- B: embed the merged articles (new, in memory only) ---
BATCH, MAX_RETRIES = 90, 6

def embed(texts, input_type):
    out = []
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i + BATCH]
        for attempt in range(MAX_RETRIES):
            try:
                r = retriever.cohere_client.embed(texts=batch, model=COHERE_MODEL,
                                                  input_type=input_type, embedding_types=["float"])
                out.extend(r.embeddings.float_)
                break
            except Exception as e:
                wait = min(60, 5 * (2 ** attempt))
                print(f"  cohere retry {attempt + 1}: {type(e).__name__} -- waiting {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError("cohere embed failed after retries")
    return out

ARTICLE_M = l2norm(np.array(embed(grouped.text.tolist(), "search_document"), dtype=np.float32))
print("article vectors:", ARTICLE_M.shape)

clause vectors: (739, 1536)


article vectors: (531, 1536)


In [4]:
# Keyword side: A/C reuse the production BM25 over clauses; B needs one over merged articles.
bm25_clause = retriever.bm25
bm25_article = BM25Okapi([tokenize(t) for t in grouped.text.tolist()], k1=BM25_K1, b=BM25_B)

query_vecs = {q.question_id: v for q, v in
              zip(evalq.itertuples(index=False), embed(evalq.question_text.tolist(), "search_query"))}
QV = {k: np.array(v, dtype=np.float32) / np.linalg.norm(v) for k, v in query_vecs.items()}
print(f"{len(QV)} query vectors")

32 query vectors


## The three retrievers

Identical fusion throughout (RRF, k=5, w=0.85, top-50 candidates per ranker) so the only variable is granularity.

In [5]:
def rrf(sem_idx, kw_idx):
    s = {}
    for rank, i in enumerate(sem_idx, 1):
        s[i] = s.get(i, 0.0) + RRF_W / (RRF_K + rank)
    for rank, i in enumerate(kw_idx, 1):
        s[i] = s.get(i, 0.0) + (1 - RRF_W) / (RRF_K + rank)
    return [i for i, _ in sorted(s.items(), key=lambda kv: -kv[1])]

def fused_order(qid, question, matrix, bm25):
    sem = np.argsort(-(matrix @ QV[qid]))[:CANDIDATE_TOP_K]
    kw = np.argsort(-bm25.get_scores(tokenize(question)))[:CANDIDATE_TOP_K]
    return rrf(list(sem), list(kw))

def slots_A(qid, question, top_k=10):
    order = fused_order(qid, question, CLAUSE_M, bm25_clause)
    out = []
    for i in order[:top_k]:
        r = chunks.iloc[i]
        out.append({"law": r["law"], "article_number": r["article_number"], "language": r["language"],
                    "clause": r["clause"], "text": r["text"], "chunk_id": r["chunk_id"]})
    return out

def slots_B(qid, question, top_k=10):
    order = fused_order(qid, question, ARTICLE_M, bm25_article)
    out = []
    for i in order[:top_k]:
        r = grouped.iloc[i]
        out.append({"law": r["law"], "article_number": r["article_number"], "language": r["language"],
                    "clause": "", "text": r["text"], "chunk_id": r["unit_id"]})
    return out

def slots_C(qid, question, top_k=10):
    """Search on clauses (same order as A), then group each article's retrieved clauses
    into a single slot -- keeping only the clauses that matched, in document order."""
    order = fused_order(qid, question, CLAUSE_M, bm25_clause)
    groups, seq = {}, []
    for i in order:
        r = chunks.iloc[i]
        key = (r["law"], r["article_number"], r["language"])
        if key not in groups:
            if len(seq) >= top_k:
                continue
            groups[key] = []
            seq.append(key)
        groups[key].append(i)
    out = []
    for key in seq[:top_k]:
        rows = [chunks.iloc[i] for i in sorted(groups[key])]
        out.append({"law": key[0], "article_number": key[1], "language": key[2],
                    "clause": ", ".join(r["clause"] for r in rows if r["clause"]),
                    "text": "\n".join(r["text"] for r in rows),
                    "chunk_id": f"{key[0]}_{key[2]}_{key[1]}", "n_clauses": len(rows)})
    return out

RETRIEVERS = {"A: current (clause slots)": slots_A,
              "B: no splitting (whole articles)": slots_B,
              "C: clauses merged per article": slots_C}

# sanity: reproduce the known baseline for q22 under A, and show what B and C do instead
q22 = eval_all[eval_all.question_id == "q22"].iloc[0]
if q22.question_id in QV:
    for name, fn in RETRIEVERS.items():
        got = [f"{s['law']}·{s['article_number']}·{s['language']}" for s in fn("q22", q22.question_text)]
        print(f"{name}\n   {got}\n")
else:
    print("q22 is cross-lingual and excluded from this run")

A: current (clause slots)
   ['ssl·71·en', 'ssl·71·en', 'ssl·62·en', 'ssl·85·en', 'ssl·71·en', 'ssl·71·en', 'ssl·71·ar', 'ssl·71·en', 'ssl·71·en', 'ssl·62·en']

B: no splitting (whole articles)
   ['ssl·90·en', 'ssl·89·ar', 'ssl·89·en', 'ssl·62·en', 'ssl·91·en', 'ssl·88·en', 'ssl·93·en', 'ssl·90·ar', 'ssl·85·en', 'ssl·67·en']

C: clauses merged per article
   ['ssl·71·en', 'ssl·62·en', 'ssl·85·en', 'ssl·71·ar', 'ssl·89·en', 'ssl·67·en', 'ssl·90·en', 'ssl·89·ar', 'ssl·93·en', 'ssl·64·ar']



## Retrieval accuracy

In [6]:
def rank_of_correct(slots, q):
    correct = set(q.answer_article_numbers)
    for pos, s in enumerate(slots, 1):
        if s["law"] == q.answer_law_code and s["language"] == q.answer_language and s["article_number"] in correct:
            return pos
    return None

def metrics(r):
    n = len(r)
    return {"recall@1": round((r == 1).sum() / n, 3),
            "recall@5": round((r.notna() & (r <= 5)).sum() / n, 3),
            "recall@10": round(r.notna().sum() / n, 3),
            "mrr": round((1 / r.dropna()).sum() / n, 3)}

slot_cache, rows = {}, []
for name, fn in RETRIEVERS.items():
    for q in evalq.itertuples(index=False):
        s = fn(q.question_id, q.question_text)
        slot_cache[(name, q.question_id)] = s
        rows.append({"config": name, "question_id": q.question_id, "rank": rank_of_correct(s, q),
                     "chars": sum(len(x["text"]) for x in s)})

retr = pd.DataFrame(rows)
summary = retr.groupby("config").apply(
    lambda d: pd.Series({**metrics(d["rank"]), "mean_chars": round(d["chars"].mean()),
                         "max_chars": d["chars"].max()}), include_groups=False
).reindex(list(RETRIEVERS))
print(summary.to_string())

                                  recall@1  recall@5  recall@10    mrr  mean_chars  max_chars
config                                                                                       
A: current (clause slots)            0.844     0.938      0.969  0.874      7038.0    21258.0
B: no splitting (whole articles)     0.875     0.969      1.000  0.904      9944.0    30271.0
C: clauses merged per article        0.844     0.938      1.000  0.879     10081.0    23755.0


In [7]:
pivot = retr.pivot(index="question_id", columns="config", values="rank")[list(RETRIEVERS)]
base = pivot["A: current (clause slots)"]
for name in list(RETRIEVERS)[1:]:
    col = pivot[name]
    print(f"{name}: {((base.isna() & col.notna()) | (col < base)).sum()} improved, "
          f"{((col.isna() & base.notna()) | (col > base)).sum()} regressed")
print("\n--- questions where the configs disagree ---")
pivot[pivot.nunique(axis=1, dropna=False) > 1]

B: no splitting (whole articles): 1 improved, 1 regressed
C: clauses merged per article: 1 improved, 0 regressed

--- questions where the configs disagree ---


config,A: current (clause slots),B: no splitting (whole articles),C: clauses merged per article
question_id,,,
q22,NaN,1.0,7.0
q23,4.0,5.0,4.0


## Generation quality

Retrieval accuracy only asks whether the right *article* appeared. It cannot see whether the generator received the clause that actually holds the answer (A can score a hit on clause (a) while the answer sits in clause (d)), nor whether bundling extra text dilutes the response. Same production generator and same independent judge as Phase 7/8.

In [8]:
from src.generation import ARTICLE_NUM_RE, get_production_generator

generator = get_production_generator()
gen_rows, answers = [], {}

for name in RETRIEVERS:
    for q in evalq.itertuples(index=False):
        slots = slot_cache[(name, q.question_id)]
        res = generator.generate(q.question_text, slots)
        answers[(name, q.question_id)] = res["answer"]
        cited = {a or b for a, b in ARTICLE_NUM_RE.findall(res["answer"])}
        available = {s["article_number"] for s in slots}
        gen_rows.append({"config": name, "question_id": q.question_id,
                         "hit_correct_article": bool(cited & set(q.answer_article_numbers)),
                         "fabricated_citation": bool(cited - available)})
    print("generated:", name)

gen = pd.DataFrame(gen_rows)
print(gen.groupby("config")[["hit_correct_article", "fabricated_citation"]].mean().round(3).to_string())

generated: A: current (clause slots)


generated: B: no splitting (whole articles)


generated: C: clauses merged per article
                                  hit_correct_article  fabricated_citation
config                                                                    
A: current (clause slots)                       0.812                0.031
B: no splitting (whole articles)                0.844                0.031
C: clauses merged per article                   0.906                0.062


In [9]:
import re as _re

from openai import OpenAI

judge_client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")
JUDGE_MODEL = "anthropic/claude-sonnet-5"

JUDGE_PROMPT = """You are grading an AI assistant's answer to a legal question about Jordanian labor/social security law. Score it on two dimensions, each 1-5.

FAITHFULNESS (1-5): Is every factual claim in the answer actually supported by the source documents below? 5 = every claim is directly backed by the documents, nothing added. 3 = mostly supported but with some unsupported/embellished detail. 1 = contains claims that contradict or aren't found anywhere in the documents.

RELEVANCY (1-5): Does the answer directly and completely address the question asked? 5 = fully addresses it, on-topic, complete. 3 = addresses it but incomplete or padded with tangential info. 1 = doesn't actually answer the question.

Question: {question}

Source documents given to the assistant:
{context}

Assistant's answer:
{answer}

Respond with ONLY this JSON, no other text:
{{"faithfulness_score": <1-5>, "faithfulness_reason": "<one sentence>", "relevancy_score": <1-5>, "relevancy_reason": "<one sentence>"}}"""

def judge(question, slots, answer, max_retries=5):
    ctx = "\n\n".join(f"[{s['law']} Article {s['article_number']} ({s['language']})]\n{s['text']}" for s in slots)
    last = None
    for attempt in range(max_retries):
        try:
            r = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": JUDGE_PROMPT.format(question=question, context=ctx, answer=answer)}])
            if r.choices:
                m = _re.search(r"\{.*\}", r.choices[0].message.content, _re.DOTALL)
                if m:
                    return json.loads(m.group(0))
        except Exception as e:
            last = e
        time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f"judge failed: {last}")

judge_rows = []
for name in RETRIEVERS:
    for q in evalq.itertuples(index=False):
        sc = judge(q.question_text, slot_cache[(name, q.question_id)], answers[(name, q.question_id)])
        judge_rows.append({"config": name, "question_id": q.question_id, **sc})
    print("judged:", name)

judged = pd.DataFrame(judge_rows)

final = (summary
         .join(gen.groupby("config")[["hit_correct_article", "fabricated_citation"]].mean().round(3))
         .join(judged.groupby("config").agg(mean_faith=("faithfulness_score", "mean"),
                                            min_faith=("faithfulness_score", "min"),
                                            mean_rel=("relevancy_score", "mean"),
                                            min_rel=("relevancy_score", "min")).round(3)))
print(final.to_string())

retr.merge(gen, on=["config", "question_id"]).merge(judged, on=["config", "question_id"]).to_csv(
    PROJECT / "eval" / "chunk_granularity_results.csv", index=False, encoding="utf-8-sig")
print("\nsaved eval/chunk_granularity_results.csv")

judged: A: current (clause slots)


judged: B: no splitting (whole articles)


judged: C: clauses merged per article
                                  recall@1  recall@5  recall@10    mrr  mean_chars  max_chars  hit_correct_article  fabricated_citation  mean_faith  min_faith  mean_rel  min_rel
config                                                                                                                                                                           
A: current (clause slots)            0.844     0.938      0.969  0.874      7038.0    21258.0                0.812                0.031       4.938          4     4.781        3
B: no splitting (whole articles)     0.875     0.969      1.000  0.904      9944.0    30271.0                0.844                0.031       4.844          3     4.875        3
C: clauses merged per article        0.844     0.938      1.000  0.879     10081.0    23755.0                0.906                0.062       4.875          3     4.844        4

saved eval/chunk_granularity_results.csv


In [10]:
print("=== any judge score <= 3 ===")
low = judged[(judged.faithfulness_score <= 3) | (judged.relevancy_score <= 3)]
for _, r in low.sort_values(["config", "question_id"]).iterrows():
    print(f"[{r.config}] {r.question_id}: faith={r.faithfulness_score} ({r.faithfulness_reason}) "
          f"| rel={r.relevancy_score} ({r.relevancy_reason})")
if low.empty:
    print("none")

=== any judge score <= 3 ===
[A: current (clause slots)] q22: faith=4 (The claims about Article 71's 0.5%/1% and 10% increases are accurate to the source, but the answer omits mention of Article 85's periodic Council of Ministers increases and Article 62's wage cap, which are also relevant and directly available.) | rel=3 (It answers the general question that pensions can increase, but focuses narrowly on disability pension provisions from Article 71 without clarifying that these are specific to disability pensions and ignoring the more general old-age pension increase mechanisms (Articles 62 and 85) present in the sources.)
[B: no splitting (whole articles)] q26: faith=3 (The quoted text from Article 88 is accurate, but the assistant's framing ('No, you cannot challenge... once notified') contradicts the actual 90-day challenge window explicitly stated in the same quoted article.) | rel=3 (The answer identifies the relevant finality rule but misrepresents the timing, failing to clearl